[![pythonista](img/pythonista.png)](https://www.pythonista.io)

# Denormalización.

## Configuración.

Ejecuta la siguiente celda antes que cualquier otra. Abre en modo de escritura la base de trabajo ```data/curso.duckdb``` y adjunta los datos del curso en sólo lectura.

In [ ]:
import duckdb

%load_ext sql
%config SqlMagic.displaylimit = 0
%config SqlMagic.displaycon = False

conexion = duckdb.connect('data/curso.duckdb')
conexion.execute("ATTACH IF NOT EXISTS 'data/thelook.duckdb' AS thelook (READ_ONLY)")
for tabla in ['distribution_centers', 'products', 'users', 'orders', 'order_items']:
    conexion.execute(f"CREATE OR REPLACE VIEW {tabla} AS SELECT * FROM thelook.main.{tabla}")
%sql conexion

## Normalizar para operar, denormalizar para analizar.

Como se vio en la lección de ```JOIN```, los sistemas transaccionales guardan cada entidad en su propia tabla (normalización) para que cada dato exista en un solo lugar y las escrituras sean rápidas y consistentes.

En analítica las necesidades son otras: se escribe poco, se lee mucho y cada consulta recorre millones de filas. Combinar cinco tablas en cada consulta es costoso y propenso a errores. La **denormalización** consiste en **combinar de antemano** las tablas relacionadas en una estructura preparada para consultarse, aunque eso implique repetir datos.

| | Modelo normalizado | Modelo denormalizado |
|---|---|---|
| Propósito | Registrar operaciones (*OLTP*) | Analizar (*OLAP*) |
| Redundancia | Mínima: cada dato en un solo lugar | Intencional: los datos del cliente se repiten en cada venta |
| Consultas | Requieren varios ```JOIN``` | Pocas o ninguna combinación |
| Escrituras | Simples y consistentes | Hay que reconstruir o actualizar la tabla |
| Almacenamiento | Menor | Mayor (en un *data warehouse* el almacenamiento es barato; el cómputo no) |

## Modelos denormalizados más comunes.

* **Esquema de estrella:** una tabla de **hechos** (eventos medibles, como cada artículo vendido) rodeada de tablas de **dimensiones** (cliente, producto, fecha) ya simplificadas. Es un punto intermedio: sigue habiendo ```JOIN```, pero sencillos y siempre del mismo tipo.
* **Tabla ancha** (*One Big Table*): los hechos y los atributos de todas sus dimensiones en una sola tabla. Es el extremo de la denormalización y resulta muy eficiente en motores columnares como *BigQuery*, porque cada consulta lee sólo las columnas que usa.

## Construcción de una tabla ancha.

Esta lección usa la misma configuración que la anterior (base de trabajo ```data/curso.duckdb```) y un esquema propio, ```modelo```, que se reinicia al comenzar.

In [ ]:
%%sql
DROP SCHEMA IF EXISTS modelo CASCADE;
CREATE SCHEMA modelo;

### Paso 1: definir la granularidad.

**Antes de denormalizar, se decide qué representa una fila.** En este caso: **un artículo vendido** (la granularidad de ```order_items```). Todas las demás tablas se combinan hacia esa granularidad:

* ```orders```, ```users```, ```products``` y ```distribution_centers``` tienen **una** fila por cada artículo, así que combinarlas no duplica filas.

### Paso 2: materializar la tabla.

In [ ]:
%%sql
CREATE TABLE modelo.ventas_detalle AS
SELECT
  -- hecho: el artículo vendido
  oi.id AS item_id,
  oi.order_id,
  oi.status AS estado_articulo,
  oi.sale_price AS precio_venta,
  oi.created_at AS fecha_venta,
  -- dimensión: orden
  o.num_of_item AS articulos_en_orden,
  o.delivered_at AS fecha_entrega,
  -- dimensión: cliente
  u.id AS cliente_id,
  u.gender AS cliente_genero,
  u.age AS cliente_edad,
  u.country AS cliente_pais,
  u.state AS cliente_estado,
  u.traffic_source AS cliente_origen,
  -- dimensión: producto
  p.id AS producto_id,
  p.name AS producto,
  p.brand AS marca,
  p.category AS categoria,
  p.department AS departamento,
  p.cost AS costo,
  -- dimensión: centro de distribución
  dc.name AS centro_distribucion
FROM order_items AS oi
JOIN orders AS o ON oi.order_id = o.order_id
JOIN users AS u ON oi.user_id = u.id
JOIN products AS p ON oi.product_id = p.id
LEFT JOIN distribution_centers AS dc
  ON p.distribution_center_id = dc.id;

Se usa ```LEFT JOIN``` en la última combinación porque algunos productos no tienen centro de distribución asignado; con ```INNER JOIN```, sus ventas desaparecerían de la tabla.

### Paso 3: validar.

Una tabla denormalizada **debe tener exactamente las mismas filas que la tabla de hechos**. Si tiene más, algún ```JOIN``` duplicó filas; si tiene menos, algún ```INNER JOIN``` descartó filas sin coincidencia.

In [ ]:
%%sql
SELECT
  (SELECT COUNT(*) FROM order_items) AS filas_origen,
  (SELECT COUNT(*) FROM modelo.ventas_detalle) AS filas_denormalizadas,
  (SELECT COUNT(DISTINCT item_id) FROM modelo.ventas_detalle) AS items_distintos;

## Consultar el modelo denormalizado.

La consulta de ventas por país y centro de distribución, que en la lección de ```JOIN``` requería cuatro tablas, ahora se resuelve sin combinaciones:

In [ ]:
%%sql
SELECT cliente_pais,
  centro_distribucion,
  ROUND(SUM(precio_venta), 2) AS ventas
FROM modelo.ventas_detalle
GROUP BY cliente_pais, centro_distribucion
ORDER BY ventas DESC
FETCH FIRST 20 ROWS ONLY;

* Margen bruto por departamento y categoría para ventas completadas.

In [ ]:
%%sql
SELECT departamento,
  categoria,
  ROUND(SUM(precio_venta), 2) AS ventas,
  ROUND(SUM(precio_venta - costo), 2) AS margen,
  ROUND(SUM(precio_venta - costo) / NULLIF(SUM(precio_venta), 0) * 100, 1) AS margen_pct
FROM modelo.ventas_detalle
WHERE estado_articulo = 'Complete'
GROUP BY departamento, categoria
ORDER BY margen DESC;

## Riesgos de la denormalización.

### Medidas de otra granularidad.

```articulos_en_orden``` es un dato de la **orden**, pero se repite en cada **artículo**. Sumarlo directamente da un resultado inflado:

In [ ]:
%%sql
SELECT
  SUM(articulos_en_orden) AS suma_incorrecta,
  COUNT(*) AS articulos_reales
FROM modelo.ventas_detalle;

Para usar medidas de una granularidad superior, primero se lleva la tabla a esa granularidad:

In [ ]:
%%sql
WITH por_orden AS (
  SELECT DISTINCT order_id, articulos_en_orden
  FROM modelo.ventas_detalle
)
SELECT SUM(articulos_en_orden) AS suma_correcta
FROM por_orden;

### Datos desactualizados.

Una tabla denormalizada es una **fotografía**: si un cliente cambia de estado en ```users```, ```ventas_detalle``` no se entera. Las estrategias habituales son:

* **Reconstrucción completa** periódica (```DROP``` + ```CREATE TABLE AS```), sencilla y adecuada para tablas medianas.
* **Carga incremental** con ```MERGE```, procesando sólo los registros nuevos o modificados.
* **Vista** en lugar de tabla, si la frescura importa más que el costo de cada consulta.

## Otra forma de denormalizar: campos anidados.

Algunos motores analíticos permiten guardar una **lista de registros dentro de una sola fila**: por ejemplo, una orden que contiene sus artículos, sin repetir los datos de la orden en cada uno. Es la forma de denormalización que recomienda *BigQuery*. Como su sintaxis varía mucho entre motores y queda fuera del *SQL* genérico, se presenta en el apéndice de *BigQuery*.

## Ejercicios.

1. Usando ```ventas_detalle```, obtén las 10 marcas con mayor margen bruto en ventas completadas.
2. Calcula el ticket promedio por país del cliente. Cuidado: el ticket es por **orden**, no por artículo.
3. Diseña (sin crearla) una tabla ancha con granularidad de **orden**: ¿qué columnas tendría? ¿Qué medidas de los artículos necesitarías agregar?
4. Crea la tabla del ejercicio anterior con ```CREATE TABLE AS``` y valida que tenga tantas filas como ```orders```.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2026.</p>